# Affiner les LLMs avec LoRA

## Objectifs d'Apprentissage

1. Apprendre à appliquer l'Adaptation de Faible Rang (LoRA) à un modèle de langue pré-entraîné
2. Maîtriser le fine-tuning d'un modèle adapté LoRA en utilisant la bibliothèque PEFT de Hugging Face
3. Comprendre comment sauvegarder et charger un modèle LoRA fine-tuné
4. Exécuter l'inférence en utilisant un modèle LoRA fine-tuné
5. Optimiser l'entraînement des modèles de langage de manière efficace en ressources

## Ensemble de Données

- **Source**: Ensemble de données "Abirate/english_quotes"
- **Utilisation**: Un échantillon de 10% de la division d'entraînement
- **Objectif**: Générer du texte basé sur des citations spécifiques

## Instructions des Tâches

1. Installer les bibliothèques nécessaires (PEFT, datasets)
2. Charger un modèle de langue pré-entraîné (bigscience/bloomz-560m) et son tokenizer
3. Charger l'ensemble de données et le prétraiter pour le modèle
4. Configurer LoRA en utilisant `LoraConfig` avec les paramètres appropriés
5. Appliquer LoRA au modèle pré-entraîné en utilisant `get_peft_model`
6. Configurer les arguments d'entraînement en utilisant `TrainingArguments`
7. Initialiser et entraîner le modèle en utilisant `Trainer`
8. Sauvegarder le modèle LoRA fine-tuné
9. Charger le modèle LoRA sauvegardé pour l'inférence en utilisant `PeftModel.from_pretrained`
10. Générer du texte en utilisant le modèle fine-tuné et le tokenizer

In [ ]:
# Task 1: Install necessary libraries
!pip install peft==0.4.0 -q
!pip install datasets -q

print("Libraries installed successfully!")

# Task 2 & 3: Load dataset, model and tokenizer
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import os

# Load pre-trained model and tokenizer
model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

print(f"Model loaded: {model_name}")

# Load dataset - Abirate/english_quotes
data = load_dataset("Abirate/english_quotes", split="train")
print(f"Dataset loaded with {len(data)} samples")

# Sample 10% of data for faster training
data = data.select(range(min(int(len(data) * 0.1), 1000)))
print(f"Using {len(data)} samples (10% sample)")

# Preprocess dataset
def preprocess(samples):
    return tokenizer(samples["quote"], truncation=True, padding=True, max_length=512)

data = data.map(preprocess, batched=True)
print("Dataset preprocessed!")

# Task 4 & 5: Configure and apply LoRA
from peft import LoraConfig, get_peft_model

# Create LoRA configuration
lora_config = LoraConfig(
    r=8,  # Rank of the adapter
    lora_alpha=32,  # Scaling factor for weight matrix
    target_modules=["query_key_value"],  # Target modules in BLOOMZ
    lora_dropout=0.1,  # Dropout rate
    bias="none",  # Do not train bias parameters
    task_type="CAUSAL_LM"
)

print("LoRA configuration created")

# Apply LoRA to the foundation model
peft_model = get_peft_model(foundation_model, lora_config)
peft_model.print_trainable_parameters()

print("LoRA applied to model!")

# Task 6 & 7: Setup training and train the model
import transformers
from transformers import TrainingArguments, Trainer

# Prepare output directory
output_directory = "./peft_lab_outputs"
os.makedirs(output_directory, exist_ok=True)

# Training arguments
training_args = TrainingArguments(
    output_dir=output_directory,
    overwrite_output_dir=False,
    auto_find_batch_size=True,
    learning_rate=3e-2,  # Higher learning rate for LoRA
    num_train_epochs=2,
    use_cpu=True,
    report_to="none",
    save_strategy="no",
    logging_steps=10
)

# Create data collator
data_collator = transformers.DataCollatorForLanguageModeling(
    tokenizer,
    mlm=False
)

# Initialize trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data,
    data_collator=data_collator
)

print("Starting training...")
trainer.train()
print("Training completed!")

# Task 8: Save the fine-tuned model
import time
time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)
print(f"Model saved to {peft_model_path}")

# Task 9 & 10: Load model for inference and generate text
from peft import PeftModel

# Load the fine-tuned LoRA model
loaded_peft_model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(model_name),
    peft_model_path,
    is_trainable=False
)

print("Fine-tuned model loaded for inference!")

# Generate text using the fine-tuned model
input_text = "Two things are infinite: "
inputs = tokenizer(input_text, return_tensors="pt")

outputs = loaded_peft_model.generate(
    **inputs,
    max_length=100,
    num_return_sequences=1,
    temperature=0.7,
    top_p=0.9
)

generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print(f"\nGenerated Text:")
print(generated_text)